In [ ]:
pip install pennylane

In [33]:
import pennylane as qml
import torch
import torch.linalg as linalg
import pennylane.numpy as np
import pandas as pd
from numpy.linalg import qr
import os

In [37]:
#dev = qml.device('default.qubit', wires=n_qubits)

def gen_unitary_qr_haar(unitary_size):
    """Generate a Haar-random matrix using the QR decomposition. Source: https://pennylane.ai/qml/demos/tutorial_haar_measure/ """
    # Step 1
    A, B = np.random.normal(size=(unitary_size, unitary_size)), np.random.normal(size=(unitary_size, unitary_size))
    Z = A + 1j * B

    # Step 2
    Q, R = qr(Z)

    # Step 3
    Lambda = np.diag([R[i, i] / np.abs(R[i, i]) for i in range(unitary_size)])

    # Step 4
    return np.dot(Q, Lambda)

'''A random vector is generated by applying random unitary matrix to qubits in zero state'''
@qml.qnode(dev, interface='torch')
def qr_haar_random_vector():
    qml.QubitUnitary(gen_unitary_qr_haar(unitary_size), wires=range(n_qubits))
    vector = qml.state()
    return vector

def gen_db_qr_haar(sort=False):
    DB = []
    for i in range(n_vectors):
      vector = qr_haar_random_vector()
      DB.append(vector.detach().numpy())

    if sort:
      DB = sorted(DB, key=lambda x: initial_probs(x, n_qubits).detach().numpy()[0], reverse=True)

    return DB

@qml.qnode(dev, interface='torch')
def quantum_circuit(weights, statevector, ansatz_func, num_layers, n_qubits):
    # Initialize the statevector
    qml.QubitStateVector(statevector, wires=range(n_qubits))

    ansatz_func(weights, n_qubits, num_layers)

    # Measure the first qubit
    return qml.probs(wires=[0])

def cost(weights, DB, n_vectors, ansatz_func, num_layers, n_qubits):
        # Compute total cost
    total_cost = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    for i in range(int(n_vectors/2)):
        transformed_sv_prob = quantum_circuit(weights, DB[i], ansatz_func, num_layers, n_qubits)
        total_cost = total_cost + (1 - transformed_sv_prob[0]) / n_vectors

    for i in range(int(n_vectors/2), int(n_vectors)):

        transformed_sv_prob = quantum_circuit(weights, DB[i], ansatz_func, num_layers, n_qubits)
        total_cost = total_cost + transformed_sv_prob[0] / n_vectors

    return total_cost

@qml.qnode(dev, interface='torch')
def initial_probs(statevector, n_qubits):
    # Initialize the statevector
    qml.QubitStateVector(statevector, wires=range(n_qubits))
    # Measure the first qubit
    return qml.probs(wires=[0])

def validate(DB, weights, n_vectors, ansatz_func, num_layers, n_qubits):
  df = pd.DataFrame(columns=['Initial Probability', 'Probability after training'])

  trained_weights = weights

  #Compare resulting and initial probabilities:
  for i in range(n_vectors):
    init_prob = initial_probs(DB[i], n_qubits)
    res_prob = quantum_circuit(trained_weights, DB[i], ansatz_func, num_layers, n_qubits)
    df.loc[i,['Initial Probability']] = init_prob.detach().numpy()[0]
    df.loc[i,['Probability after training']] = res_prob.detach().numpy()[0]
  return df

def three_BasicEntanglerLayers(weights, n_qubits, num_layers):
  for i in range(num_layers):
    if i % 3 == 0:
      qml.BasicEntanglerLayers(weights=weights[i : i+1,:], wires=range(n_qubits), rotation=qml.RX)
    elif i % 3 == 1:
      qml.BasicEntanglerLayers(weights=weights[i : i+1,:], wires=range(n_qubits), rotation=qml.RZ)
    elif i % 3 == 2:
      qml.BasicEntanglerLayers(weights=weights[i : i+1,:], wires=range(n_qubits), rotation=qml.RY)

def ZXZ_BasicEntanglerLayers(weights, n_qubits, num_layers):
  for i in range(num_layers):
    if i % 3 == 0:
      qml.BasicEntanglerLayers(weights=weights[i : i+1,:], wires=range(n_qubits), rotation=qml.RZ)
    elif i % 3 == 1:
      qml.BasicEntanglerLayers(weights=weights[i : i+1,:], wires=range(n_qubits), rotation=qml.RX)
    elif i % 3 == 2:
      qml.BasicEntanglerLayers(weights=weights[i : i+1,:], wires=range(n_qubits), rotation=qml.RZ)

def StronglyEntanglingLayers(weights, n_qubits, num_layers):
  qml.StronglyEntanglingLayers(weights=weights, wires=range(n_qubits))

def save_weights_tensor(weights, n_qubits, num_layers):
    # Create directory structure based on parameters
    directory = f"weights/n_qubits_{n_qubits}/num_layers_{num_layers}"
    os.makedirs(directory, exist_ok=True)

    # Define file path
    file_path = os.path.join(directory, "weights.pt")

    # Save the tensor
    torch.save(weights, file_path)

    print(f"Weights saved to {file_path}")


In [ ]:
DB = gen_db_qr_haar()

In [ ]:
# Training of ansatz parameters for fixed n_qubits and num_layers
n_qubits = 5
unitary_size = 2**n_qubits
n_vectors = 2**n_qubits
num_layers = 16
ansatz_func = StronglyEntanglingLayers

shape = qml.StronglyEntanglingLayers.shape(n_layers=num_layers, n_wires=n_qubits)

weights = 2*np.pi*torch.rand(shape, dtype=torch.float64) #Initialise parameters - rnadom values from uniform distribution in range [0;2pi]
weights = torch.tensor(weights, dtype=torch.float64, requires_grad=True)

optimizer = torch.optim.Adam([weights], lr=0.01)

# Number of optimization steps (epochs)
epochs = 500

# For early stopping:
previous_loss = None
early_stopping_threshold = 0.00001

# Optimization loop
for epoch in range(epochs):

    def closure():
      # Zero gradients
      optimizer.zero_grad()

      # Forward pass: compute cost function
      output = cost(weights, DB, n_vectors, ansatz_func, num_layers, n_qubits)

      # Compute loss
      loss = output

      # Backward pass: compute gradient of the loss
      loss.backward()

      return loss

    # Update parameters
    loss = optimizer.step(closure)

    # Print progress
    if epoch % 1 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')
        if epoch % 50 == 0:
          print(f'{validate(DB, weights, n_vectors, ansatz_func, num_layers, n_qubits)}')
          print(weights.grad)

    # Check for early stopping
    if previous_loss is not None:
        loss_difference = abs(previous_loss - loss.item())
        if loss_difference < early_stopping_threshold:
            print(f"Early stopping at epoch {epoch+1} due to small loss difference: {loss_difference}")
            break

    previous_loss = loss.item()

In [44]:
#Loss after training 16 layers of StronglyEntanglingLayers for 5 qubits
loss.item()

0.17196417689693952

In [ ]:
# Training of ansatz parameters for different n_qubits and num_layers
n_qubits_list = [3,4,5]
num_layers_list = [2,4,6,8]
ansatz_func = StronglyEntanglingLayers

df_all = pd.DataFrame(columns=['Loss value', 'n_qubits', 'num_layers'])

ind = 0
for n_qubits in n_qubits_list:
  unitary_size = 2**n_qubits
  n_vectors = 2**n_qubits

  dev = qml.device('default.qubit', wires=n_qubits)


  #Reinitialise function that contain a decorator
  @qml.qnode(dev, interface='torch')
  def qr_haar_random_vector():
    qml.QubitUnitary(gen_unitary_qr_haar(unitary_size), wires=range(n_qubits))
    vector = qml.state()
    return vector

  @qml.qnode(dev, interface='torch')
  def quantum_circuit(weights, statevector, ansatz_func, num_layers, n_qubits):
      # Initialize the statevector
      qml.QubitStateVector(statevector, wires=range(n_qubits))

      ansatz_func(weights, n_qubits, num_layers)

      # Measure the first qubit
      return qml.probs(wires=[0])

  @qml.qnode(dev, interface='torch')
  def initial_probs(statevector, n_qubits):
      # Initialize the statevector
      qml.QubitStateVector(statevector, wires=range(n_qubits))
      # Measure the first qubit
      return qml.probs(wires=[0])

  DB = gen_db_qr_haar()
  print(DB)

  for num_layers in num_layers_list:
    ind += 1
    df_all.loc[ind,['n_qubits']] = n_qubits
    df_all.loc[ind,['num_layers']] = num_layers

    shape = qml.StronglyEntanglingLayers.shape(n_layers=num_layers, n_wires=n_qubits)

    weights = 2*np.pi*torch.rand(shape, dtype=torch.float64)
    weights = torch.tensor(weights, dtype=torch.float64, requires_grad=True)

    optimizer = torch.optim.Adam([weights], lr=0.01)

    # Number of optimization steps (epochs)
    epochs = 500

    # For early stopping:
    previous_loss = None
    early_stopping_threshold = 0.00001

    # Optimization loop
    for epoch in range(epochs):

        def closure():
          # Zero gradients
          optimizer.zero_grad()

          # Forward pass: compute cost function
          output = cost(weights, DB, n_vectors, ansatz_func, num_layers, n_qubits)

          # Compute loss
          loss = output

          # Backward pass: compute gradient of the loss
          loss.backward()

          return loss

        # Update parameters
        loss = optimizer.step(closure)

        # Print progress
        if epoch % 1 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')
            if epoch % 50 == 0:
              print(f'{validate(DB, weights, n_vectors, ansatz_func, num_layers, n_qubits)}')
              print(weights.grad)

        # Check for early stopping
        if previous_loss is not None:
            loss_difference = abs(previous_loss - loss.item())
            if loss_difference < early_stopping_threshold:
                print(f"Early stopping at epoch {epoch+1} due to small loss difference: {loss_difference}")
                break

        previous_loss = loss.item()

    df_all.loc[ind,['Loss value']] = loss.item()
    save_weights_tensor(weights, n_qubits, num_layers)
    print(df_all)

In [43]:
df_all

,Loss value,n_qubits,num_layers
1,0.177519,3,2
2,0.122927,3,4
3,0.117393,3,6
4,0.116742,3,8
5,0.365277,4,2
6,0.217717,4,4
7,0.188531,4,6
8,0.156611,4,8
9,0.389643,5,2
10,0.318041,5,4
